# KING2 — Merge LoRA + Base Model
دمج محول LoRA مع Qwen2.5-3B-Instructor ورفع الناتج إلى Hugging Face

In [ ]:
import os
HF_TOKEN = os.getenv("HF_TOKEN")  # set via environment, do NOT hardcode
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
LORA_ADAPTER = "RASHID778/king2-qwen2.5-3b"
OUTPUT_REPO = "RASHID778/king2-qwen2.5-3b-merged"

In [ ]:
!pip install -q transformers peft torch accelerate huggingface_hub

In [ ]:
import torch
from huggingface_hub import HfApi, login
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN,
)
print("Base loaded")

print("Loading LoRA...")
model = PeftModel.from_pretrained(model, LORA_ADAPTER, token=HF_TOKEN)
print("LoRA loaded")

In [ ]:
print("Merging... (takes 1-2 min)")
merged = model.merge_and_unload()
print("Merge complete!")

In [ ]:
print("Saving locally...")
merged.save_pretrained("./king2-merged")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.save_pretrained("./king2-merged")
print("Saved")

In [ ]:
print("Uploading to HF...")
api = HfApi()
api.create_repo(OUTPUT_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)
api.upload_folder(folder_path="./king2-merged", repo_id=OUTPUT_REPO, token=HF_TOKEN)
print(f"Uploaded: https://huggingface.co/{OUTPUT_REPO}")